# 02_heart_reference

Selected source cells for review and reconstruction.

Source: `Heart_combined_samples/Reference_merge_heart.ipynb`

Requires original scRNA-seq Seurat reference, TMS reference and h5ad. Reference() conversion to reference_rctd_heart.rds was not found.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 1 (one-based); original execution count 1.

In [ ]:
library(Seurat)

heart_scrna <- readRDS("heart_ageing_scRNAseq_reference_clean.rds")
heart_10x   <- readRDS("heart_ageing_10x_clean.rds")

DefaultAssay(heart_scrna) <- "RNA"
DefaultAssay(heart_10x)   <- "RNA"

# gene overlap 確認（重要）
genes_use <- intersect(rownames(heart_scrna), rownames(heart_10x))
length(genes_use)


Source cell 2 (one-based); original execution count 2.

In [ ]:
heart_scrna <- subset(heart_scrna, features = genes_use)
heart_10x   <- subset(heart_10x,   features = genes_use)


Source cell 3 (one-based); original execution count 3.

In [ ]:
heart_scrna$origin <- "scRNAseq"
heart_10x$origin   <- "10x_ageing"


Source cell 7 (one-based); original execution count 8.

In [ ]:
library(zellkonverter)
library(SingleCellExperiment)

sce_heart <- readH5AD("heart_ageing_10x_str.h5ad")


Source cell 10 (one-based); original execution count 11.

In [ ]:
celltype_vec <- colData(sce_heart)$cell_type
names(celltype_vec) <- colnames(sce_heart)

Source cell 11 (one-based); original execution count 12.

In [ ]:
heart_10x <- readRDS("heart_ageing_10x_clean.rds")

# barcode を揃える
common_cells <- intersect(colnames(heart_10x), names(celltype_vec))

heart_10x <- heart_10x[, common_cells]
heart_10x$cell_type <- celltype_vec[common_cells]


Source cell 13 (one-based); original execution count 14.

In [ ]:
saveRDS(
  heart_10x,
  file = "heart_ageing_10x_reference_annotated.rds"
)

Source cell 15 (one-based); original execution count 47.

In [ ]:
vasc <- subset(
  heart_scrna,
  subset = celltype == "Vascular cell"
)

Source cell 16 (one-based); original execution count 55.

In [ ]:
DefaultAssay(vasc) <- "RNA"

vasc <- NormalizeData(vasc, verbose = FALSE)
vasc <- FindVariableFeatures(vasc, verbose = FALSE)
vasc <- ScaleData(vasc, verbose = FALSE)
vasc <- RunPCA(vasc, verbose = FALSE)

vasc <- FindNeighbors(vasc, dims = 1:20)
vasc <- FindClusters(vasc, resolution = 0.3)
vasc <- RunUMAP(vasc, dims = 1:20)


Source cell 21 (one-based); original execution count 59.

In [ ]:
# 初期化
vasc$Cellname_refined <- "Endothelial"

# Lymphatic
vasc$Cellname_refined[vasc$seurat_clusters == "4"] <- "Lymphatic endothelial"

# Pericyte
vasc$Cellname_refined[vasc$seurat_clusters == "6"] <- "Pericyte"

# cluster 7 は明示的に別扱い（後で除外）
vasc$Cellname_refined[vasc$seurat_clusters == "7"] <- "Vascular_noise"


Source cell 30 (one-based); original execution count 81.

In [ ]:
# vasc はすでに作成済みで、vasc$Cellname_refined が
# Endothelial / Lymphatic endothelial / Pericyte / Vascular_noise になっている前提

# 0) 念のため確認（vasc 側）
table(vasc$Cellname_refined, useNA = "ifany")

# 1) heart_scrna に新しい列を作る（まず celltype をコピー）
heart_scrna$Cellname_final <- as.character(heart_scrna$celltype)

# 2) vasc の refined を heart_scrna に上書き（Cells名で対応付け）
#    ここで factor を避けて character で入れる
map_refined <- setNames(as.character(vasc$Cellname_refined), Cells(vasc))

# heart_scrna のうち vasc に該当する細胞だけ置換
idx <- Cells(heart_scrna) %in% names(map_refined)
heart_scrna$Cellname_final[idx] <- map_refined[Cells(heart_scrna)[idx]]

# 3) 確認
cat("=== celltype ===\n")
print(table(heart_scrna$celltype))

cat("\n=== Cellname_final ===\n")
print(table(heart_scrna$Cellname_final, useNA = "ifany"))

# 4) さらに sanity check：vasc部分だけを抜き出して見たい場合
cat("\n=== Cellname_final within Vascular cell ===\n")
print(table(heart_scrna$Cellname_final[heart_scrna$celltype == "Vascular cell"], useNA = "ifany"))


Source cell 31 (one-based); original execution count 82.

In [ ]:
# 1) 除外対象を定義
exclude_cells <- c(
  "Fibroblast",
  "Neutrophil",
  "Mast cell",
  "Vascular_noise"
)

# 2) サブセット作成
heart_scrna_ref_clean <- subset(
  heart_scrna,
  subset = !Cellname_final %in% exclude_cells
)

# 3) 確認
cat("=== Cellname_final (clean) ===\n")
print(table(heart_scrna_ref_clean$Cellname_final))

cat("\nTotal cells before:", ncol(heart_scrna), "\n")
cat("Total cells after :", ncol(heart_scrna_ref_clean), "\n")


Source cell 32 (one-based); original execution count 83.

In [ ]:
saveRDS(
  heart_scrna_ref_clean,
  file = "heart_scrna_reference_clean_step1.rds"
)

cat("✅ 保存完了: heart_scrna_reference_clean_step1.rds\n")


Source cell 34 (one-based); original execution count 98.

In [ ]:
# 明示的に設定
DefaultAssay(heart_scrna_ref_clean) <- "RNA"
DefaultAssay(heart_10x) <- "RNA"

# 確認
DefaultAssay(heart_scrna_ref_clean)
DefaultAssay(heart_10x)

# 念のため Assay 一覧も確認
Assays(heart_scrna_ref_clean)
Assays(heart_10x)


Source cell 37 (one-based); original execution count 103.

In [ ]:
# 明示的に origin を付与
heart_scrna_ref_clean$origin <- "scRNAseq"
heart_10x$origin             <- "10x"

# 確認
table(heart_scrna_ref_clean$origin)
table(heart_10x$origin)


Source cell 38 (one-based); original execution count 104.

In [ ]:
heart_merged <- merge(
  x = heart_scrna_ref_clean,
  y = heart_10x,
  add.cell.ids = c("scRNAseq", "10x"),
  project = "Heart_reference_merged"
)


Source cell 40 (one-based); original execution count 106.

In [ ]:
library(future)
plan(multicore, workers = 16)
options(future.globals.maxSize = 200 * 1024^3)

DefaultAssay(heart_merged) <- "RNA"

heart_merged <- SCTransform(
  heart_merged,
  method = "glmGamPoi",
  vars.to.regress = NULL,
  verbose = TRUE
)

Source cell 42 (one-based); original execution count 108.

In [ ]:
heart_merged <- RunPCA(
  heart_merged,
  npcs = 50,
  verbose = FALSE
)

Source cell 44 (one-based); original execution count 111.

In [ ]:
library(harmony)

Source cell 45 (one-based); original execution count 114.

In [ ]:
# PCA まで済んでいる前提
heart_merged <- IntegrateLayers(
  object = heart_merged,
  method = HarmonyIntegration,
  orig.reduction = "pca",
  new.reduction = "harmony",
  group.by.vars = "origin",
  dims = 1:30
)


Source cell 47 (one-based); original execution count 116.

In [ ]:
heart_merged <- RunUMAP(
  heart_merged,
  reduction = "harmony",
  dims = 1:30
)

heart_merged <- FindNeighbors(
  heart_merged,
  reduction = "harmony",
  dims = 1:30
)

heart_merged <- FindClusters(
  heart_merged,
  resolution = 0.5
)


Source cell 48 (one-based); original execution count 118.

In [ ]:
# 念のため character に
heart_merged$Cellname_final <- as.character(heart_merged$Cellname_final)

# 10x 側だけ cell_type を代入
heart_merged$Cellname_final[
  heart_merged$origin == "10x"
] <- as.character(
  heart_merged$cell_type[heart_merged$origin == "10x"]
)

# factor に戻す
heart_merged$Cellname_final <- factor(heart_merged$Cellname_final)


Source cell 52 (one-based); original execution count 126.

In [ ]:
map_cellname <- c(
  "B cell / Plasma cell"                = "B cell lineage",
  "cardiac muscle cell"                 = "Cardiac muscle cell",
  "cardiac neuron"                      = "Cardiac neuron",
  "endocardial cell"                    = "Endocardial cell",
  "Endothelial"                         = "Endothelial cell",
  "endothelial cell of coronary artery" = "Endothelial cell",
  "erythrocyte"                         = "Erythroid cell",
  "fibroblast of cardiac tissue"        = "Fibroblast",
  "leukocyte"                           = "Macrophage / Monocyte / DC",
  "Lymphatic endothelial"               = "Lymphatic endothelial",
  "Macrophage / Monocyte / DC"          = "Macrophage / Monocyte / DC",
  "mast cell"                           = "Mast cell",
  "Pericyte"                            = "Pericyte",
  "smooth muscle cell"                  = "VSMC",
  "T cell / NK cell / ILC"              = "T cell lineage"
)


Source cell 53 (one-based); original execution count 127.

In [ ]:
# meta.data を一度取り出す
md <- heart_merged@meta.data

# Cellname_final を使って新しい列を作る
md$Cellname <- map_cellname[
  as.character(md$Cellname_final)
]

# rownames が cell barcode であることを確認
stopifnot(identical(rownames(md), colnames(heart_merged)))

# meta.data に戻す
heart_merged@meta.data <- md


Source cell 56 (one-based); original execution count 130.

In [ ]:
saveRDS(
  heart_merged,
  file = "heart_reference_merged_sct_harmony_step1.rds"
)

cat("✅ 保存完了: heart_reference_merged_sct_harmony_step1.rds\n")
